# Computation G — confinement as the monogamy obstruction

**It from Bit via Gödel, Paper 1** · companion to `sec:confinement` · the centrepiece, made runnable

The framework's account of why colour-charged quarks cannot be isolated is **distance-free**. A
baryon's three colour (C) qubits form a maximal GHZ state, the colour singlet. By monogamy of
entanglement — the budget $\tau_3 + (\text{bipartite terms}) \le 1$ — a maximal GHZ state has
$\tau_3 = 1$, which forces **every** bipartite term to zero: the triple has no residual
entanglement available for any external partner.

The key point, and the one to read carefully: **this has nothing to do with distance.** $\tau_3$
is a property of the *state* (its amplitudes / reduced density matrix), with no spatial argument.
A GHZ state has $\tau_3 = 1$ whether the qubits sit in one trap or three galaxies. Confinement
here is not a budget that "drains with separation" — entanglement is not distance-dependent. It is
the flat structural fact that **a colour singlet cannot acquire an external colour partner without
ceasing to be a singlet.** This notebook demonstrates exactly that.

*Scope.* This shows the structural obstruction (you cannot isolate a quark). It does **not** derive
the distance-dependent confinement potential V(r) — that requires the emergent-distance machinery
and is the open problem of `sec:confinement`.

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, partial_trace

Y = np.array([[0,-1j],[1j,0]])
def concurrence(rho):
    R = rho @ np.kron(Y,Y) @ rho.conj() @ np.kron(Y,Y)
    ev = np.sort(np.sqrt(np.clip(np.linalg.eigvals(R).real, 0, None)))[::-1]
    return max(0.0, ev[0]-ev[1]-ev[2]-ev[3])

def one_tangle(psi3, q):           # tau_{q(rest)} = 2(1 - Tr rho_q^2)
    rho = partial_trace(Statevector(psi3), [i for i in range(3) if i != q]).data
    return float(2*(1 - np.trace(rho@rho).real))

def two_tangle(psi3, a, b):        # squared concurrence of the (a,b) reduced pair
    rho = partial_trace(Statevector(psi3), [i for i in range(3) if i not in (a,b)]).data
    return concurrence(rho)**2

def tau3(psi3):                    # CKW: tau_1(23) - C^2_12 - C^2_13
    return one_tangle(psi3,0) - two_tangle(psi3,0,1) - two_tangle(psi3,0,2)

def ket(b): v=np.zeros(8,complex); v[int(b,2)]=1; return v
GHZ = (ket('000')+ket('111'))/np.sqrt(2)   # the colour singlet (C-qubits)
W   = (ket('001')+ket('010')+ket('100'))/np.sqrt(3)
prod= ket('000')

## The colour singlet saturates the budget

GHZ: $\tau_3 = 1$ and **both** two-tangles zero — all entanglement is irreducibly tripartite, none
is left over for an external partner. (W is shown for contrast: by the CKW measure W has *zero*
genuine three-tangle, its entanglement is entirely in the pairs — which is why the W/lepton sector
is *not* confined in this way.)

In [2]:
print(f"{'state':8s} {'tau1(23)':>9s} {'C^2_12':>8s} {'C^2_13':>8s} {'tau3':>8s}")
for nm,s in [("GHZ",GHZ),("W",W),("product",prod)]:
    print(f"{nm:8s} {one_tangle(s,0):9.4f} {two_tangle(s,0,1):8.4f} "
          f"{two_tangle(s,0,2):8.4f} {tau3(s):8.4f}")
print("\nGHZ: tau3=1, residual two-tangles=0 -> NO entanglement free for an external partner.")
print("This is the colour singlet. The statement is about the state; no distance appears.")

state     tau1(23)   C^2_12   C^2_13     tau3
GHZ         1.0000   0.0000   0.0000   1.0000
W           0.8889   0.4444   0.4444  -0.0000
product     0.0000   0.0000   0.0000   0.0000

GHZ: tau3=1, residual two-tangles=0 -> NO entanglement free for an external partner.
This is the colour singlet. The statement is about the state; no distance appears.


## Trying to free a quark: the obstruction in action

Take the colour singlet GHZ$(q_0q_1q_2)$ with an external qubit $q_3$ in $|0\rangle$, and apply a
genuine entangling rotation between one quark ($q_2$) and the external qubit. Watch the two numbers
move **in lockstep**: as external entanglement $C(q_2, q_3)$ rises, the GHZ-fidelity of the triple
falls. The quark can acquire an outside partner *only* by surrendering its place in the singlet.
There is no configuration with both a full singlet and an external bond — that is confinement, with
no distance anywhere in the argument.

In [3]:
from qiskit.quantum_info import Operator

def partial_iswap(a):                      # iSWAP-family coupling on two qubits
    c, s = np.cos(a), np.sin(a)
    return np.array([[1,0,0,0],[0,c,-1j*s,0],[0,-1j*s,c,0],[0,0,0,1]], complex)

def state4(a):
    qc = QuantumCircuit(4)
    qc.h(0); qc.cx(0,1); qc.cx(1,2)        # colour singlet GHZ on q0,q1,q2 (x |0>_3)
    qc.unitary(Operator(partial_iswap(a)), [2,3])   # move entanglement onto q2-external
    return qc

print(f"{'coupling a':>10s} {'GHZ-fid(012)':>13s} {'C(q2,ext)':>11s}")
for a in [0.0, 0.2, 0.4, 0.6, 0.785]:
    sv = Statevector(state4(a))
    rho012 = partial_trace(sv, [3]).data
    fid = (GHZ.conj() @ rho012 @ GHZ).real
    rho23 = partial_trace(sv, [0,1]).data
    print(f"{a:10.3f} {fid:13.4f} {concurrence(rho23):11.4f}")
print("\nExternal entanglement C(q2,3) RISES as singlet fidelity FALLS, in lockstep.")
print("A quark bonds outside ONLY by surrendering its place in the singlet: that is")
print("confinement -- no external colour partner is available to a colour singlet.")

coupling a  GHZ-fid(012)   C(q2,ext)
     0.000        1.0000      0.0000
     0.200        0.9802      0.1947
     0.400        0.9226      0.3587
     0.600        0.8330      0.4660
     0.785        0.7288      0.5000

External entanglement C(q2,3) RISES as singlet fidelity FALLS, in lockstep.
A quark bonds outside ONLY by surrendering its place in the singlet: that is
confinement -- no external colour partner is available to a colour singlet.


## Running on hardware

Unlike the sampler-based notebooks (A, C, D, E), the quantities here --- the three-tangle $\tau_3$
and the concurrences --- are functions of the *reduced density matrices*, not direct measurement
outcomes. On hardware they must be obtained by **state tomography**: measure the colour-singlet in
every Pauli basis, reconstruct $\rho$ by linear inversion, and compute $\tau_3$ from it. Set
`USE_HARDWARE = True` to run this on an IBM backend; `False` (default) reconstructs from an exact
simulator through the identical tomography code, so the path is verified end-to-end.

**What to expect on hardware.** Tomographic $\tau_3$ will read *below* the exact value of $1$ ---
readout error, gate noise, and the linear-inversion reconstruction all pull it down, and 27 basis
circuits accumulate noise. A hardware $\tau_3 \approx 0.8$--$0.95$ still demonstrates the structural
point (the singlet is dominated by genuine tripartite entanglement, with little residual bipartite
tangle available externally); the exact saturation $\tau_3 = 1$ is the noise-free statement. This is
a heavier protocol than a single sampler call, by the nature of what is being measured.

In [4]:
USE_HARDWARE = False    # flip to True to run tomography on IBM hardware
SHOTS = 4096

from itertools import product
from collections import defaultdict
from qiskit.quantum_info import DensityMatrix

PAULI = {'I':np.eye(2), 'X':np.array([[0,1],[1,0]]),
         'Y':np.array([[0,-1j],[1j,0]]), 'Z':np.array([[1,0],[0,-1]])}

def singlet_prep():                      # the colour singlet: GHZ on 3 C-qubits
    qc = QuantumCircuit(3); qc.h(0); qc.cx(0,1); qc.cx(1,2); return qc

def tomography_circuits():
    circs, labels = [], []
    for bset in product('XYZ', repeat=3):
        qc = singlet_prep()
        for q,b in enumerate(bset):
            if b=='X': qc.h(q)
            elif b=='Y': qc.sdg(q); qc.h(q)
        qc.measure_all(); circs.append(qc); labels.append(bset)
    return circs, labels

circs, labels = tomography_circuits()     # 27 Pauli-basis circuits

if USE_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    service = QiskitRuntimeService()
    backend = service.least_busy(simulator=False, operational=True)
    print("backend:", backend.name)
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    isa = [pm.run(c) for c in circs]
    results = Sampler(mode=backend).run(isa, shots=SHOTS).result()
else:
    from qiskit.primitives import StatevectorSampler
    results = StatevectorSampler().run(circs, shots=SHOTS).result()

# reconstruct all 3-qubit Pauli expectations, then rho by linear inversion
acc = defaultdict(list)
for r, bset in zip(results, labels):
    counts = r.data.meas.get_counts(); tot = sum(counts.values())
    for mask in product([0,1], repeat=3):
        if not any(mask): continue
        P = tuple(bset[q] if mask[q] else 'I' for q in range(3))
        e = sum(((-1)**sum(1 for q in range(3) if mask[q] and b[::-1][q]=='1'))*c
                for b,c in counts.items())/tot
        acc[P].append(e)
exp = {P: float(np.mean(v)) for P,v in acc.items()}; exp[('I','I','I')] = 1.0

rho = np.zeros((8,8), complex)
for P,e in exp.items():
    M = PAULI[P[0]]
    for k in (1,2): M = np.kron(M, PAULI[P[k]])
    rho += e*M
rho /= 8

def ptr(keep):
    return partial_trace(DensityMatrix(rho), [q for q in range(3) if q not in keep]).data
t1 = float(2*(1 - np.trace(ptr([0])@ptr([0])).real).real)
c12 = concurrence(ptr([0,1])); c13 = concurrence(ptr([0,2]))
tau3_hw = t1 - c12**2 - c13**2
print(f"\ntomographic tau1(23) = {t1:.3f}   C12 = {c12:.3f}   C13 = {c13:.3f}")
print(f"tomographic tau3      = {tau3_hw:.3f}   (exact: 1.000; below 1 on noisy hardware)")
print("Residual external-available tangle (C12^2 + C13^2) =",
      f"{c12**2 + c13**2:.3f}  (exact: 0 -- nothing available to an external partner)")


tomographic tau1(23) = 1.000   C12 = 0.002   C13 = 0.000
tomographic tau3      = 1.000   (exact: 1.000; below 1 on noisy hardware)
Residual external-available tangle (C12^2 + C13^2) = 0.000  (exact: 0 -- nothing available to an external partner)


## What to vary, and what this does and does not show

Try other couplings (`rxx`/`ryy` angles, or couple a different quark to the external qubit) — the
lockstep is generic: any external entanglement is paid for in singlet fidelity, because the budget
is saturated. This is the **structural** obstruction, and it is exact and distance-free.

What it does **not** show: the measured distance-dependent potential V(r) ~ σr. That is the open
problem of `sec:confinement` — because distance is emergent in this framework, deriving an
r-dependent potential from this distance-free constraint requires the relational/epistemic-spacetime
machinery, and is where confinement touches Paper 2. This notebook is the *why-confined*; the
*how-strongly-with-distance* is owed.